# Capstone Project - Stock Price Analysis using Regression
**Guided Lab**

> ⚠️ **Educational use only.** Stock prices are very hard to predict. This lab teaches the *regression workflow*, not how to trade. Nothing here is financial advice.

## Learning objectives
By the end of this lab you will be able to:
1. Import stock price data from a free online source into a pandas DataFrame
2. Clean and prepare time-series data and engineer features
3. Visualise prices, returns and relationships, and draw insights
4. Build **simple** and **multiple linear regression** models with scikit-learn
5. Evaluate models properly on time-ordered data, against a naive baseline
6. Save a trained model and use it to predict on new data

## How this lab works
- Each **Task** has instructions and an **empty code cell** for you to complete.
- Stuck? Click **"💡 Click here to reveal the solution"** under the task. Try first.
- Run cells **in order**. Later tasks use variables created in earlier ones, so use the exact variable names given in each task.

## Roadmap
| Part | Topic | Tasks |
|---|---|---|
| 1 | Data import | 1-2 |
| 2 | Data preparation | 3-4 |
| 3 | Visuals and insights | 5-7 |
| 4 | Modelling | 8-13 |
| 5 | Save the model and predict on new data | 14-15 |
| 6 | Reflection and extensions | questions |

## Setup (run this cell, nothing to code)
Installs libraries, imports them, and defines a few helpers. `make_synthetic_prices()` creates offline practice data in case your network blocks the data download in Task 1.

In [ ]:
%pip install -q yfinance scikit-learn matplotlib pandas joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams["figure.figsize"] = (10, 4)

# ---- Lab configuration (feel free to change the ticker later) ----
TICKER = "AAPL"          # Apple. Try MSFT, TSLA, INFY.NS, RELIANCE.NS ...
START  = "2019-01-01"

# ---- Helper 1: evaluation metrics in one line ----
def evaluate(name, y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE %": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        "R2": r2_score(y_true, y_pred),
    }

# ---- Helper 2: offline fallback data (simulated prices, NOT real) ----
def make_synthetic_prices(n_days=1500, seed=42, start_price=100):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=n_days)
    ret = rng.normal(0.0005, 0.015, n_days)
    close = start_price * np.exp(np.cumsum(ret))
    open_ = close * (1 + rng.normal(0, 0.003, n_days))
    high = np.maximum(open_, close) * (1 + np.abs(rng.normal(0, 0.005, n_days)))
    low = np.minimum(open_, close) * (1 - np.abs(rng.normal(0, 0.005, n_days)))
    volume = rng.integers(20_000_000, 80_000_000, n_days)
    return pd.DataFrame({"Open": open_, "High": high, "Low": low,
                         "Close": close, "Volume": volume},
                        index=pd.Index(dates, name="Date"))

print("Setup complete. Ticker:", TICKER)

---
# Part 1: Data Import
**Data source:** [Yahoo Finance](https://finance.yahoo.com/) via the free `yfinance` Python library. No account or API key needed.

### Task 1: Download stock prices into a DataFrame

Using `yfinance`, download daily prices for `TICKER` starting from `START` and store them in a DataFrame called **`df`**.

**Steps**
1. `import yfinance as yf`
2. Call `yf.download(TICKER, start=START, auto_adjust=True, progress=False)`
3. If the columns come back as a MultiIndex, flatten them with `df.columns.get_level_values(0)`
4. Make sure the index is a `DatetimeIndex` and keep only `Open, High, Low, Close, Volume`
5. **If the download fails** (network restrictions), use `df = make_synthetic_prices()` as a fallback

**Checkpoint:** `df` has 5 columns and well over 1000 rows (about 7 years of trading days).

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
try:
    import yfinance as yf
    df = yf.download(TICKER, start=START, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):          # newer yfinance returns a MultiIndex
        df.columns = df.columns.get_level_values(0)
    assert len(df) > 100, "Too few rows returned"
    print("Loaded real data from Yahoo Finance")
except Exception as e:
    print("Download failed:", e, "\n-> using offline simulated data instead")
    df = make_synthetic_prices()

df.index = pd.to_datetime(df.index)
df = df[["Open", "High", "Low", "Close", "Volume"]]
df.head()
```

</details>

### Task 2: Explore the structure of the data

Answer these questions with code:
- How many rows and columns are there? (`df.shape`)
- What is the first and last date in the data?
- What are the data types and are there missing values? (`df.info()`)
- Show summary statistics (`df.describe()`)
- Display the first 5 and last 5 rows

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
print("Shape:", df.shape)
print("Date range:", df.index.min().date(), "to", df.index.max().date())
print()
df.info()
display(df.describe().round(2))
display(df.head())
display(df.tail())
```

</details>

---
# Part 2: Data Preparation

### Task 3: Clean the data

Real data is messy. Check and fix the following, keeping the result in **`df`**:
1. Count missing values per column
2. Count duplicate dates and remove them
3. Sort the index in ascending date order
4. Fill any gaps using forward-fill (`ffill`), then drop any rows still empty

**Why forward-fill?** On a missing day, the best guess for a price is the last known price. We never use *future* values to fill the past.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
print("Missing values:\n", df.isna().sum())
print("\nDuplicate dates:", df.index.duplicated().sum())

df = df[~df.index.duplicated()].sort_index()
df = df.ffill().dropna()

print("\nAfter cleaning ->", df.shape, "| missing:", int(df.isna().sum().sum()))
```

</details>

### Task 4: Feature engineering and target creation

We want to predict **tomorrow's closing price**. Create a new DataFrame **`df_feat`** (copy of `df`) with these columns:

| Column | Definition |
|---|---|
| `Return` | daily % change of `Close` (`pct_change()`) |
| `Close_lag1` | yesterday's close (`shift(1)`) |
| `Close_lag2` | close two days ago (`shift(2)`) |
| `SMA_5` | 5-day moving average of `Close` |
| `SMA_20` | 20-day moving average of `Close` |
| `Volatility_10` | 10-day rolling std-dev of `Return` |
| `Volume_chg` | daily % change in `Volume` |
| **`Target`** | **tomorrow's close** (`Close.shift(-1)`) |

Then:
- Define `FEATURES = ["Close","Close_lag1","Close_lag2","SMA_5","SMA_20","Return","Volatility_10","Volume_chg"]`
- Save the **latest row's features** (the one with no known Target yet) as **`latest_row`** (a 1-row DataFrame). We will predict tomorrow from it in Task 15.
- Replace `inf` values with NaN, then drop rows with NaN to create **`df_model`**

**Think:** Why do the first ~20 rows and the last row get dropped?

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
df_feat = df.copy()
df_feat["Return"]        = df_feat["Close"].pct_change()
df_feat["Close_lag1"]    = df_feat["Close"].shift(1)
df_feat["Close_lag2"]    = df_feat["Close"].shift(2)
df_feat["SMA_5"]         = df_feat["Close"].rolling(5).mean()
df_feat["SMA_20"]        = df_feat["Close"].rolling(20).mean()
df_feat["Volatility_10"] = df_feat["Return"].rolling(10).std()
df_feat["Volume_chg"]    = df_feat["Volume"].pct_change()
df_feat["Target"]        = df_feat["Close"].shift(-1)     # tomorrow's close

FEATURES = ["Close", "Close_lag1", "Close_lag2", "SMA_5", "SMA_20",
            "Return", "Volatility_10", "Volume_chg"]

latest_row = df_feat[FEATURES].iloc[[-1]]                 # today's features, tomorrow unknown

df_feat = df_feat.replace([np.inf, -np.inf], np.nan)
df_model = df_feat.dropna()

print("Rows before:", len(df_feat), "| after dropna:", len(df_model))
df_model.head()
```

**Answer to the think question:** rolling windows (e.g. 20-day average) and lags need history, so the first rows are NaN. The last row has no *tomorrow* yet, so its Target is NaN.

</details>

---
# Part 3: Visuals and Insights

### Task 5: Visualise price trend and volume

Create a figure with **two stacked plots** sharing the x-axis:
1. Top: `Close` price with `SMA_5` and `SMA_20` overlaid (add a legend)
2. Bottom: trading `Volume` as a bar or area chart

Use `plt.subplots(2, 1, sharex=True, figsize=(11, 7))`.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(11, 7),
                               gridspec_kw={"height_ratios": [3, 1]})
ax1.plot(df_feat.index, df_feat["Close"], label="Close", linewidth=1.2)
ax1.plot(df_feat.index, df_feat["SMA_5"], label="SMA 5", alpha=0.8)
ax1.plot(df_feat.index, df_feat["SMA_20"], label="SMA 20", alpha=0.8)
ax1.set_title(f"{TICKER} - Close price and moving averages")
ax1.set_ylabel("Price"); ax1.legend()

ax2.fill_between(df_feat.index, df_feat["Volume"], alpha=0.5)
ax2.set_ylabel("Volume"); ax2.set_xlabel("Date")
plt.tight_layout(); plt.show()
```

</details>

### Task 6: Visualise returns and relationships

Create **three** visuals:
1. A **histogram** of daily `Return` (50 bins)
2. A **correlation heatmap** of `FEATURES + ["Target"]` (use `df_model.corr()` and `plt.imshow`)
3. A **scatter plot** of today's `Close` (x) vs tomorrow's `Target` (y)

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# 1. Return distribution
axes[0].hist(df_model["Return"], bins=50)
axes[0].set_title("Distribution of daily returns")
axes[0].set_xlabel("Return")

# 2. Correlation heatmap
cols = FEATURES + ["Target"]
corr = df_model[cols].corr()
im = axes[1].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(cols))); axes[1].set_xticklabels(cols, rotation=60, ha="right")
axes[1].set_yticks(range(len(cols))); axes[1].set_yticklabels(cols)
axes[1].set_title("Correlation heatmap")
plt.colorbar(im, ax=axes[1])

# 3. Today vs tomorrow
axes[2].scatter(df_model["Close"], df_model["Target"], s=6, alpha=0.5)
axes[2].set_title("Today's close vs tomorrow's close")
axes[2].set_xlabel("Close (today)"); axes[2].set_ylabel("Target (tomorrow)")

plt.tight_layout(); plt.show()
print("Correlation of each feature with Target:")
print(corr["Target"].drop("Target").sort_values(ascending=False).round(3))
```

</details>

### Task 7: Write your data insights

Look at your visuals and write **at least 3 observations**. Prompts:
- Is there a long-term trend? Any sudden drops or spikes?
- Are daily returns roughly bell-shaped? Are there extreme days (fat tails)?
- Which features correlate most strongly with tomorrow's price? Are some features nearly identical to each other (multicollinearity)?
- Does the Close vs Target scatter look linear?

*Write your answer in the cell below (double-click to edit).*

✏️ **My insights:**

- 
- 
-

<details>
<summary><b>💡 Click here to reveal a sample answer</b></summary>

**Sample insights** (yours will vary with the ticker and dates):
- Price shows a long-term trend with periods of sharp movement. The moving averages smooth the noise, and SMA_5 reacts faster than SMA_20.
- Daily returns are centred near 0 and roughly bell-shaped, but with a few extreme days (fat tails).
- `Close`, `Close_lag1`, `Close_lag2`, `SMA_5` and `SMA_20` all correlate about 0.99+ with `Target`. They are also highly correlated with *each other* (**multicollinearity**), so individual coefficients may be unstable.
- Today vs tomorrow is almost a straight line: prices are **autocorrelated**. This makes a simple regression look excellent, so we must compare against a naive baseline later.
- `Return`, `Volatility_10` and `Volume_chg` show weak correlation with the price level, but may carry small incremental signal.

</details>

---
# Part 4: Modelling

### Task 8: Time-based train/test split

For time series we **must not shuffle** the data. If we did, the model would train on the future and be tested on the past (*data leakage*), and the results would be unrealistically good.

Split `df_model` **chronologically**: first 80% of rows = train, last 20% = test. Create:
`X_train, y_train, X_test, y_test` (features = `FEATURES`, target = `"Target"`).

Print the date ranges of both sets.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
split = int(len(df_model) * 0.8)
train, test = df_model.iloc[:split], df_model.iloc[split:]

X_train, y_train = train[FEATURES], train["Target"]
X_test,  y_test  = test[FEATURES],  test["Target"]

print(f"Train: {train.index.min().date()} -> {train.index.max().date()}  ({len(train)} rows)")
print(f"Test : {test.index.min().date()} -> {test.index.max().date()}  ({len(test)} rows)")
```

</details>

### Task 9: Baseline model: the naive forecast

Before building a model, build the simplest possible benchmark: **"tomorrow's price = today's price"**.

1. Create `y_pred_naive` = `X_test["Close"]`
2. Create a list `results = []` and append `evaluate("Naive baseline", y_test, y_pred_naive)`

Any model worth using must **beat this baseline**.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
y_pred_naive = X_test["Close"]

results = []
results.append(evaluate("Naive baseline (tomorrow = today)", y_test, y_pred_naive))
pd.DataFrame(results).round(3)
```

</details>

### Task 10: Simple linear regression

Fit a **simple** linear regression using only **one** feature, `Close`, to predict `Target`.

1. Train `LinearRegression()` on `X_train[["Close"]]` as **`model_simple`**
2. Print the intercept and slope, and write the equation
3. Predict on the test set (`y_pred_simple`) and append its metrics to `results`

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
model_simple = LinearRegression()
model_simple.fit(X_train[["Close"]], y_train)

b0, b1 = model_simple.intercept_, model_simple.coef_[0]
print(f"Target = {b0:.3f} + {b1:.4f} x Close")

y_pred_simple = model_simple.predict(X_test[["Close"]])
results.append(evaluate("Simple LR (Close)", y_test, y_pred_simple))
pd.DataFrame(results).round(3)
```

</details>

### Task 11: Multiple linear regression

Now use **all** `FEATURES`. Build a scikit-learn **pipeline** that standardises the features and then fits a linear regression:

`make_pipeline(StandardScaler(), LinearRegression())`

1. Train it on `X_train, y_train` as **`model_multi`**
2. Show the **standardised coefficients** in a table sorted by absolute size
3. Predict (`y_pred_multi`) and append the metrics to `results`

**Why a pipeline?** The scaler is learnt from training data only (no leakage) and is saved together with the model.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
model_multi = make_pipeline(StandardScaler(), LinearRegression())
model_multi.fit(X_train, y_train)

coefs = pd.DataFrame({
    "feature": FEATURES,
    "std_coefficient": model_multi[-1].coef_,
})
coefs["abs"] = coefs["std_coefficient"].abs()
display(coefs.sort_values("abs", ascending=False).drop(columns="abs").round(3))
print("Intercept:", round(model_multi[-1].intercept_, 3))

y_pred_multi = model_multi.predict(X_test)
results.append(evaluate("Multiple LR (8 features)", y_test, y_pred_multi))
```

</details>

### Task 12: Compare all models

Display `results` as a DataFrame rounded to 3 decimals. Then answer:
- Which model has the lowest RMSE and MAPE?
- Did the regression models beat the naive baseline?

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
comparison = pd.DataFrame(results).set_index("Model").round(3)
display(comparison)
print("Best RMSE:", comparison["RMSE"].idxmin())
```

</details>

### Task 13: Diagnose the best model

Create two plots for `model_multi`:
1. **Actual vs Predicted over time** on the test set (line plot)
2. **Residuals over time** (`y_test - y_pred_multi`) with a horizontal line at 0

Check: are residuals centred on 0 and free of obvious patterns? Is the error larger in volatile periods?

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
residuals = y_test - y_pred_multi

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(y_test.index, y_test, label="Actual")
axes[0].plot(y_test.index, y_pred_multi, label="Predicted", alpha=0.8)
axes[0].set_title("Test set: Actual vs Predicted next-day close"); axes[0].legend()

axes[1].plot(residuals.index, residuals, linewidth=0.8)
axes[1].axhline(0, color="red")
axes[1].set_title("Residuals (Actual - Predicted)")
plt.tight_layout(); plt.show()

print("Mean residual:", round(residuals.mean(), 3), "| Std:", round(residuals.std(), 3))
```

</details>

### Task 14: Write your modelling insights

Summarise your findings. Prompts:
- How good is R²? Is that impressive given the naive baseline?
- How much better (or not) is multiple LR vs simple LR vs baseline in RMSE/MAPE terms?
- Which features carry the biggest coefficients? Can you trust them given multicollinearity?
- What would a typical prediction error mean in **rupees/dollars** for a trader?
- What limitations does a linear model on price level have?

*Write your answer in the cell below (double-click to edit).*

✏️ **My insights:**

- 
- 
-

<details>
<summary><b>💡 Click here to reveal a sample answer</b></summary>

**Sample insights** (numbers depend on ticker and period):
- R² is typically very high (0.95+) for price-level models, **but this is misleading**: prices are strongly autocorrelated, so even the naive "tomorrow = today" forecast scores a high R².
- The fair test is the **baseline comparison**. The regression models usually match the naive baseline closely, with only small gains (or none). This is evidence of how hard day-ahead prediction is.
- The largest standardised coefficients usually land on `Close` and the lags/moving averages, which are almost the same variable. Because of multicollinearity, their individual signs and sizes should not be over-interpreted.
- A typical error (RMSE/MAE) is a few currency units, or about 1-2% (MAPE). That is similar in size to a normal daily move, so it is not enough to trade profitably on its own.
- Limitations: linear model, price level non-stationary, no news or macro information, errors grow in volatile periods, a single train/test split.

</details>

---
# Part 5: Save the Model and Predict on New Data

### Task 15: Save the trained model

Save `model_multi` to disk with `joblib`, together with the feature list and ticker (so anyone loading it knows what inputs it expects).

- File name: `stock_regression_model.joblib`
- Save a **dictionary**: `{"model": model_multi, "features": FEATURES, "ticker": TICKER}`
- Confirm the file exists with `os.path.exists` and print its size

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
import os

artifact = {"model": model_multi, "features": FEATURES, "ticker": TICKER}
joblib.dump(artifact, "stock_regression_model.joblib")

print("Saved:", os.path.exists("stock_regression_model.joblib"),
      "| size:", os.path.getsize("stock_regression_model.joblib"), "bytes")
```

</details>

### Task 16: Load the model and predict on new data

Pretend you are a different user in a **new session**.

1. Load `stock_regression_model.joblib` into `loaded` and extract the `model` and `features`
2. **Back-test on "new" data:** predict the last 5 rows of the test set and show Actual vs Predicted side by side with the error
3. **Predict tomorrow:** use `latest_row` from Task 4 to forecast the next trading day's close
4. **What-if scenario:** copy `latest_row`, increase `Close` and `SMA_5` by 2%, and predict again. What does the model say?

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python
loaded = joblib.load("stock_regression_model.joblib")
model, feats = loaded["model"], loaded["features"]

# 2. Back-test on the most recent 5 known days
new_X = X_test.tail(5)
compare = pd.DataFrame({
    "Actual": y_test.tail(5),
    "Predicted": model.predict(new_X[feats]),
})
compare["Error"] = compare["Actual"] - compare["Predicted"]
display(compare.round(2))

# 3. Predict tomorrow
tomorrow = model.predict(latest_row[feats])[0]
print(f"\nLast close ({df.index[-1].date()}): {df['Close'].iloc[-1]:.2f}")
print(f"Predicted next close: {tomorrow:.2f}")

# 4. What-if scenario
whatif = latest_row.copy()
whatif["Close"] *= 1.02
whatif["SMA_5"] *= 1.02
print(f"What-if (+2% price): {model.predict(whatif[feats])[0]:.2f}")
```

</details>

---
# Part 6: Reflection Questions and Extensions

### Reflection questions
1. Why did we split the data chronologically instead of randomly? What would go wrong otherwise?
2. A friend says "my model has R² = 0.99, it's brilliant!" What question do you ask first?
3. Name two features in this lab that are nearly duplicates. What problem does that cause?
4. Why is forward-fill acceptable for missing prices but backward-fill is not?
5. Why must the `StandardScaler` be fit on the training data only?
6. Your model predicts a price of 187.40 and the actual is 190.10. Compute the error, absolute error and percentage error.
7. What would change if you predicted **next-day return** instead of next-day price? Would R² go up or down? Why?

### Extension challenges
- 🔹 **Change ticker:** try `MSFT`, `TSLA` or an Indian stock like `RELIANCE.NS`. Does the model rank differently against its baseline?
- 🔹 **Predict returns:** set `Target = Return.shift(-1)` and rebuild. Compare R² with the price-level model.
- 🔹 **Regularisation:** swap `LinearRegression` for `Ridge` or `Lasso`. Does it tame the multicollinearity?
- 🔹 **Better validation:** use `sklearn.model_selection.TimeSeriesSplit` for walk-forward cross-validation instead of a single split.
- 🔹 **More features:** add RSI, MACD, day-of-week, or lagged returns. Does test RMSE improve?
- 🔹 **Longer horizon:** predict the close 5 days ahead (`shift(-5)`). How does error grow?

### Key takeaways
| Lesson | Why it matters |
|---|---|
| Always build a naive baseline | High R² on trending data can be an illusion |
| Split by time, never shuffle | Prevents data leakage |
| Watch multicollinearity | Coefficients become unstable and hard to interpret |
| Pipeline = model + preprocessing | Saved model works on raw new data |
| Test errors in real units | MAE/MAPE tell you what the error means in money |

---
*End of Capstone. Congratulations on completing the regression workflow end to end.*